# Testing MCP Client with Typescript MCP Server on Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host an existing STDIO MCP server (Model Context Protocol) server using the Amazon Bedrock AgentCore runtime environment.

### Tutorial Details

| Information         | Details                                                   |
|:--------------------|:----------------------------------------------------------|
| Tutorial type       | Hosting existing STDIO MCP server                             |
| Tool type           | MCP server                                                |
| Tutorial components | Hosting STDIO MCP server on AgentCore Runtime        |
| Tutorial vertical   | Cross-vertical                                            |
| Example complexity  | Easy                                                      |


### Tutorial Overview

1. The AgentCore Runtime authentication will use Amazon Cognito to provide JWT tokens for accessing our deployed MCP server.

2. The mcp server is an existing MCP server using STDIO. We will use https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server for this example

3. The mcp client is written in python.
   _Note the mcp client can be written in any language._

## Prerequisites

To execute this tutorial you will need:
- Python 3.10+ (MCP client)
- Docker (for containerization)  
- Amazon ECR (Elastic Container Registry) for storing Docker images  
- AWS account with access to Bedrock AgentCore  
- MCP (Model Context Protocol) library
- (Optional) MCP Inspector (`npx @modelcontextprotocol/inspector`)

In [ ]:
#!uv add -r requirements.txt --active

## Understanding MCP (Model Context Protocol)

MCP is a protocol that allows AI models to securely access external data and tools. Key concepts:

* **Tools**: Functions that the AI can call to perform actions
* **Prompts**: Prompts allow servers to provide structured messages and instructions for interacting with LLM
* **Streamable HTTP**: Transport protocol used by AgentCore Runtime
* **Session Isolation**: Each client gets isolated sessions via `Mcp-Session-Id` header
* **Stateless Operation**: Servers must support stateless operation for scalability

AgentCore Runtime expects MCP servers to be hosted on `0.0.0.0:8000/mcp` as the default path.

The MCP protocol can use multiple transports, the most common one being STDIO. In this tutorial we show how to take an existing STDIO server and expose it via HTTP in AgentCore Runtime so that it can be made available to users without requiring installation on local machines. It also makes the tool available to Agents without requiring installation on the local agent environment.

## Step 1: Setting up Amazon Cognito for Authentication

AgentCore Runtime requires authentication. We'll use Amazon Cognito to provide JWT tokens for accessing our deployed MCP server.

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import setup_cognito_user_pool

In [ ]:
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

In [ ]:
CLIENT_ID = cognito_config.get('client_id', None)
DISCOVERY_URL = cognito_config.get('discovery_url', None)

In [ ]:
import boto3

sess = boto3.Session(region_name='us-east-1')
client = sess.client('sts')
ACCOUNT = client.get_caller_identity()['Account']

## Step 2: Running the proxy MCP server

Let's build a docker container that contain our STDIO MCP server and the proxy that exposes it as a streamable HTTP server.
We use [FastMCP proxy](https://gofastmcp.com/servers/proxy) functionality to expose the local STDIO server as HTTP.

In [ ]:
%%writefile main.py
from fastmcp import FastMCP
from fastmcp.client.transports import StdioTransport
from fastmcp.server.proxy import ProxyClient
from starlette.responses import JSONResponse
import os

# Create a proxy directly from a config dictionary
transport = StdioTransport(
    command="uv",
    args=["run", "awslabs.aws-documentation-mcp-server"],
)

# Create a proxy to the configured server (auto-creates ProxyClient)
proxy = FastMCP.as_proxy(ProxyClient(transport), name="Proxy", stateless_http=True)


@proxy.custom_route("/ping", ["GET"])
def ping(req):
    return JSONResponse({"status": "healthy"})


# Run the proxy with stdio transport for local access
if __name__ == "__main__":
    # Default to localhost for security, but allow override via environment variable
    # In Docker, set HOST=0.0.0.0 to bind to all interfaces
    host = os.environ.get("HOST", "127.0.0.1")
    port = int(os.environ.get("PORT", "8000"))
    proxy.run(transport="streamable-http", host=host, port=port)


In a terminal run:

```bash
docker build -t mcp-stdio-proxy .
```

Once build we can run it and test it with the MCP Inspector (`npx @modelcontextprotocol/inspector`):

```bash
docker run --rm -p 8000:8000 mcp-stdio-proxy
```


## Step 3: MCP Server Deployment to AgentCore

We leverage the starter toolkit to deploy this project, by manually creating a `.bedrock-agentcore.yaml` file.

In [ ]:
config = f"""
default_agent: mcp_stdio
agents:
  mcp_stdio:
    name: mcp_stdio
    entrypoint: main.py
    platform: linux/arm64
    container_runtime: docker
    aws:
      execution_role: 
      execution_role_auto_create: true
      account: "{ACCOUNT}"
      region: us-east-1
      ecr_repository:
      ecr_auto_create: true
      network_configuration:
        network_mode: PUBLIC
      protocol_configuration:
        server_protocol: MCP
      observability:
        enabled: true
    bedrock_agentcore:
      agent_id: 
      agent_arn: 
      agent_session_id: null
    codebuild:
      project_name: null
      execution_role: null
      source_bucket: null
    memory:
      mode: NO_MEMORY
    authorizer_configuration: 
      customJWTAuthorizer: 
        allowedClients:
          - {CLIENT_ID}
        discoveryUrl: {DISCOVERY_URL}
    request_header_configuration: null
    oauth_configuration: null
"""

with open(".bedrock_agentcore.yaml", "w") as f:
    f.write(config)

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.runtime.launch  import launch_bedrock_agentcore

In [ ]:
from pathlib import Path

result = launch_bedrock_agentcore(config_path=Path('.bedrock_agentcore.yaml'), use_codebuild=False, auto_update_on_conflict=True)

## Step 4: Creating Testing Client

Now let's create a client to test our deployed MCP server. This client will retrieve the necessary credentials from AWS and connect to the deployed server:

In [ ]:
agent_arn = result.agent_arn
   
encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
mcp_url = f"https://bedrock-agentcore.{sess.region_name}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
print(mcp_url)

In [ ]:

from utils import reauthenticate_user
bearer_token = reauthenticate_user(cognito_config['client_id'])
print(bearer_token)

You can use the above information to test the client using MCP Inspector

In [ ]:

import sys
import traceback


from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

import json

async def list_tools():
    agent_arn = result.agent_arn
    bearer_token = reauthenticate_user(cognito_config['client_id'])
    print(bearer_token)
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{sess.region_name}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()
                
                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        print(traceback.format_exc(e))
        sys.exit(1)

## Step 7: Testing Your Deployed MCP Server

Let's test our deployed MCP server using the remote client:

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
await list_tools()

## Step 8: Invoking MCP Tools Remotely

Now let's create an enhanced client that not only lists tools but also invokes them to demonstrate the full MCP functionality:

In [ ]:
import sys

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def call_tools():
    agent_arn = result.agent_arn
    bearer_token = reauthenticate_user(cognito_config['client_id'])
    print(bearer_token)
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{sess.region_name}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConnecting to: {mcp_url}")
    print("Headers configured ✓")
    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")
                
                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()
                
                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)
                
                try:
                    print("\nTesting read_documentation")
                    add_result = await session.call_tool(
                        name="read_documentation",
                        arguments={"max_length": 5000, "url": "https://docs.aws.amazon.com/lambda/latest/dg/lambda-invocation.html"}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\nTesting search")
                    add_result = await session.call_tool(
                        name="search",
                        arguments={"limit": 10, "search_phrase": "agentcore runtime"}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")
                
                
                print("\n✅ MCP tool testing completed!")
                
    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        sys.exit(1)


## Test Tool Invocation

Let's test our MCP tools by actually invoking them:

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
await call_tools()

## Next Steps

Now that you have successfully deployed an STDIO MCP server to AgentCore Runtime, you can:

1. **Try another STDIO MCP server**: Try to host other STDIO MCP servers and transform them in company wide accessible tools
2. **Configure other OAuth2 providers**: Implement custom JWT authorizers

# 🎉 Congratulations!

You have successfully:

✅ **Hosted an existing STDIO MCP server** using FastMCP proxy  
✅ **Set up authentication** with Amazon Cognito  
✅ **Deployed to AWS** using AgentCore Runtime  
✅ **Tested MCP tools remotely** with proper authentication  
✅ **Learned MCP concepts** and best practices  

Your MCP server is now running on Amazon Bedrock AgentCore Runtime and ready for production use!